# Personalized Education Assistant
### A sequential three-agent CrewAI system: learning materials, a quiz, and project ideas

Give the system a short list of **topics** and an **expertise level** (`beginner`, `intermediate`,
or `advanced`). Three agents then run in order:

| # | Agent | Task | Structured output |
|---|---|---|---|
| 1 | **Learning Material Agent** | Curate videos, articles and exercises for the topics (web search via Serper) | `LearningMaterials` |
| 2 | **Quiz Creator Agent** | Write a quiz that tests the curated material | `Quiz` |
| 3 | **Project Idea Agent** | Recommend practical projects for the expertise level, using a **custom tool** | `ProjectIdeas` |

Each task's output is a validated **Pydantic model**, so the result is data you can use, not just prose.

**Keys.** `OPENAI_API_KEY` is required. `SERPER_API_KEY` enables live web search for the first agent;
without it the agent still runs, recommending resources from its own knowledge.

---
## 1. Install

The version is pinned so the notebook keeps working when CrewAI changes its API.
`crewai[tools]` brings in `crewai-tools`, which provides `SerperDevTool`.

In [33]:
!pip install -q "crewai[tools]==1.15.33"

---
## 2. API keys

Keys are looked up in this order, and the first one found wins:

1. **Pasted directly below** - fill in `OPENAI_API_KEY` / `SERPER_API_KEY` in the cell if you prefer not to use Colab Secrets (remove them before sharing the notebook).
2. **Colab Secrets** - the key icon in the left sidebar. Add `OPENAI_API_KEY` (and optionally `SERPER_API_KEY`) and turn on *Notebook access*.
3. **Environment variable** - for running outside Colab.
4. **Prompt** - `getpass` asks for the required key without echoing it.

Only `True`/`False` is printed, never a key value.

In [34]:
import os
from getpass import getpass

# --- Option 1: paste keys here (leave empty to use Colab Secrets / env / prompt) ---
OPENAI_API_KEY = ""
SERPER_API_KEY = ""   # optional - enables web search for learning materials


def load_key(name: str, pasted: str, required: bool) -> str:
    if pasted.strip():
        return pasted.strip()
    try:
        from google.colab import userdata  # only exists on Colab
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get(name, "")
    if value or not required:
        return value
    return getpass(f"Enter {name}: ")


os.environ["OPENAI_API_KEY"] = load_key("OPENAI_API_KEY", OPENAI_API_KEY, required=True)
serper = load_key("SERPER_API_KEY", SERPER_API_KEY, required=False)
if serper:
    os.environ["SERPER_API_KEY"] = serper

# Keep the run quiet: no CrewAI tracing prompt or telemetry.
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ["OTEL_SDK_DISABLED"] = "true"

print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("SERPER_API_KEY loaded:", bool(os.environ.get("SERPER_API_KEY")), "(optional - web search)")

OPENAI_API_KEY loaded: True
SERPER_API_KEY loaded: True (optional - web search)


---
## 3. Learner profile

The only user input. Change the topics and level here, then run everything below.

The expertise level is checked against the three allowed values **before** anything reaches an
agent - it drives the custom tool's output, so it must be one of the values the tool knows.
Topics are capped in count and length so the prompt stays small and predictable.

In [35]:
TOPICS = ["Python decorators", "SQL joins"]
EXPERTISE_LEVEL = "beginner"          # beginner | intermediate | advanced

EXPERTISE_LEVELS = ("beginner", "intermediate", "advanced")
MAX_TOPICS, MAX_TOPIC_LEN = 5, 60

EXPERTISE_LEVEL = EXPERTISE_LEVEL.strip().lower()
if EXPERTISE_LEVEL not in EXPERTISE_LEVELS:
    raise ValueError(f"EXPERTISE_LEVEL must be one of {EXPERTISE_LEVELS}, got {EXPERTISE_LEVEL!r}")

TOPICS = [t.strip() for t in TOPICS if t and t.strip()]
if not TOPICS or len(TOPICS) > MAX_TOPICS:
    raise ValueError(f"TOPICS must contain 1 to {MAX_TOPICS} entries, got {len(TOPICS)}")
for t in TOPICS:
    if len(t) > MAX_TOPIC_LEN or any(ch in t for ch in "{}\n"):
        raise ValueError(f"Invalid topic: {t!r} (max {MAX_TOPIC_LEN} chars, no braces or newlines)")

print("Topics:", TOPICS)
print("Level :", EXPERTISE_LEVEL)

Topics: ['Python decorators', 'SQL joins']
Level : beginner


---
## 4. Structured outputs (Pydantic models)

One model per task. CrewAI passes the model's JSON schema to the LLM and validates the reply
against it, so a missing field or wrong type is caught by Pydantic rather than discovered later.
`Field(description=...)` text is visible to the model and steers what it fills in.

In [36]:
from typing import List, Literal
from pydantic import BaseModel, Field


# --- Task 1: learning materials -------------------------------------------
class Resource(BaseModel):
    title: str = Field(description="Title of the video or article")
    url: str = Field(description="Direct link to the resource")
    why: str = Field(description="One sentence on why this fits the learner")


class Exercise(BaseModel):
    title: str
    instructions: str = Field(description="What the learner should do, in 1-3 sentences")


class LearningMaterials(BaseModel):
    topics: List[str]
    expertise_level: str
    videos: List[Resource] = Field(description="2-4 videos across the topics")
    articles: List[Resource] = Field(description="2-4 articles or docs pages across the topics")
    exercises: List[Exercise] = Field(description="2-4 hands-on exercises")


# --- Task 2: quiz ----------------------------------------------------------
class QuizQuestion(BaseModel):
    question: str
    options: List[str] = Field(description="Exactly 4 answer options, labelled A) B) C) D)")
    correct_answer: str = Field(description="The letter of the correct option: A, B, C or D")
    explanation: str = Field(description="Why the correct answer is correct")


class Quiz(BaseModel):
    title: str
    questions: List[QuizQuestion] = Field(description="5 questions grounded in the learning materials")


# --- Task 3: project ideas -------------------------------------------------
class ProjectIdea(BaseModel):
    title: str
    description: str = Field(description="What the learner will build, 2-3 sentences")
    skills_practiced: List[str]
    estimated_hours: int = Field(description="Rough effort to complete")


class ProjectIdeas(BaseModel):
    expertise_level: str
    projects: List[ProjectIdea] = Field(description="3 projects suited to the expertise level")


print("Models defined:", [m.__name__ for m in (LearningMaterials, Quiz, ProjectIdeas)])

Models defined: ['LearningMaterials', 'Quiz', 'ProjectIdeas']


---
## 5. Custom tool: Project Suggestion Tool

A CrewAI tool is a class that subclasses `BaseTool` and implements `_run`. `args_schema` is a
Pydantic model describing the inputs - CrewAI shows that schema to the agent, so the agent knows
exactly what arguments to pass, and Pydantic rejects anything else (e.g. an expertise level that
is not one of the three allowed values).

The tool is deterministic: it maps the expertise level to a **project template** (scope, constraints
and what a finished project should demonstrate) and returns it for the requested topic. The agent
then uses that template to write the three concrete project ideas. Keeping the level-to-scope
rule in code means a "beginner" never gets a distributed-systems project because the model felt
ambitious.

In [37]:
from crewai.tools import BaseTool

PROJECT_TEMPLATES = {
    "beginner": {
        "scope": "A single small script or notebook (under ~100 lines) with one clear job.",
        "constraints": "No frameworks or external services; standard library or one well-known package only.",
        "should_demonstrate": "Correct use of the core concept, readable code, and one simple test or sanity check.",
        "typical_hours": "2-4",
    },
    "intermediate": {
        "scope": "A multi-file project or notebook with 2-3 interacting components and real input data.",
        "constraints": "Use at least one external library or API; include error handling and a few unit tests.",
        "should_demonstrate": "Applying the concept to a realistic problem, handling edge cases, and documenting usage in a README.",
        "typical_hours": "6-12",
    },
    "advanced": {
        "scope": "A deployable or benchmarkable system with measurable results.",
        "constraints": "Must include performance or correctness evaluation, configuration, and automated tests.",
        "should_demonstrate": "Design trade-offs, optimisation or scaling of the concept, and a short write-up of results.",
        "typical_hours": "15-30",
    },
}


class ProjectSuggestionInput(BaseModel):
    topic: str = Field(description="The topic the project should practise")
    expertise_level: Literal["beginner", "intermediate", "advanced"] = Field(
        description="The learner's expertise level"
    )


class ProjectSuggestionTool(BaseTool):
    name: str = "Project Suggestion Tool"
    description: str = (
        "Returns the project scope, constraints and success criteria appropriate for a learner's "
        "expertise level and topic. Call it once per topic, then write concrete project ideas that fit."
    )
    args_schema: type[BaseModel] = ProjectSuggestionInput

    def _run(self, topic: str, expertise_level: str) -> str:
        t = PROJECT_TEMPLATES[expertise_level]
        return (
            f"Project template for '{topic}' at {expertise_level} level:\n"
            f"- Scope: {t['scope']}\n"
            f"- Constraints: {t['constraints']}\n"
            f"- Should demonstrate: {t['should_demonstrate']}\n"
            f"- Typical effort: {t['typical_hours']} hours"
        )


project_tool = ProjectSuggestionTool()

# Quick check: the tool works on its own, and rejects a bad level.
print(project_tool.run(topic="SQL joins", expertise_level="beginner"))
try:
    project_tool.run(topic="SQL joins", expertise_level="expert")
    print("\nWARNING: invalid level was not rejected")
except Exception as e:
    print("\nRejected invalid level as expected:", type(e).__name__)

Project template for 'SQL joins' at beginner level:
- Scope: A single small script or notebook (under ~100 lines) with one clear job.
- Constraints: No frameworks or external services; standard library or one well-known package only.
- Should demonstrate: Correct use of the core concept, readable code, and one simple test or sanity check.
- Typical effort: 2-4 hours

Rejected invalid level as expected: ValueError


---
## 6. Agents

Three agents, one per task. `allow_delegation=False` keeps the sequence fixed - the task order
*is* the design. `max_iter` bounds the reasoning loop so a confused agent cannot spend tokens
forever.

The Learning Material Agent gets `SerperDevTool` (Google search via Serper) when a key is present.
The Project Idea Agent gets the custom tool from section 5. The Quiz Creator needs no tool: its
input is the previous task's output.

In [38]:
from crewai import Agent, Crew, LLM, Process, Task

MODEL = "gpt-4o-mini"
llm = LLM(model=MODEL, temperature=0.3)

# Disable external search tool temporarily to bypass opentelemetry/requests conflicts in Colab environment
search_tools = []
print("Web search enabled:", bool(search_tools))

learning_material_agent = Agent(
    role="Learning Material Curator",
    goal="Find high-quality videos, articles and exercises that teach {topics} at a {expertise_level} level",
    backstory=(
        "You are an experienced technical educator who knows the best free learning resources "
        "on the web. You prefer official documentation, well-known tutorial sites and reputable "
        "video channels, and you always explain why a resource suits the learner's level."
    ),
    tools=search_tools,
    llm=llm,
    allow_delegation=False,
    max_iter=8,
    verbose=True,
)

quiz_creator_agent = Agent(
    role="Quiz Creator",
    goal="Write clear multiple-choice questions that check understanding of the curated learning materials",
    backstory=(
        "You design assessments for online courses. Your questions test understanding rather than "
        "memorisation, have exactly one correct answer, and come with a short explanation."
    ),
    llm=llm,
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

project_idea_agent = Agent(
    role="Project Mentor",
    goal="Recommend practical projects that match a {expertise_level} learner studying {topics}",
    backstory=(
        "You mentor learners by giving them hands-on projects. You always consult the Project "
        "Suggestion Tool to size a project correctly for the learner's level before describing it."
    ),
    tools=[project_tool],
    llm=llm,
    allow_delegation=False,
    max_iter=8,
    verbose=True,
)
print("Agents ready.")

Web search enabled: False
Agents ready.


---
## 7. Tasks

`{topics}` and `{expertise_level}` are filled in from the `inputs` dict at kickoff. `context`
passes a previous task's output into a later task - this is what makes the quiz test *the curated
material* rather than the topic in general. `output_pydantic` ties each task to its model from
section 4.

In [39]:
curate_task = Task(
    description=(
        "Curate learning materials for a {expertise_level} learner on these topics: {topics}.\n"
        "Find 2-4 videos, 2-4 articles or documentation pages, and write 2-4 short hands-on exercises. "
        "Cover every topic at least once. Use real, specific URLs. "
        "If web search is available, use it to confirm resources exist."
    ),
    expected_output="A LearningMaterials object with videos, articles and exercises for every topic.",
    agent=learning_material_agent,
    output_pydantic=LearningMaterials,
)

quiz_task = Task(
    description=(
        "Using the curated learning materials, write a 5-question multiple-choice quiz for a "
        "{expertise_level} learner on {topics}. Each question has exactly 4 options labelled "
        "A) to D), one correct letter, and a one-sentence explanation. "
        "Spread the questions across all topics."
    ),
    expected_output="A Quiz object with a title and 5 questions.",
    agent=quiz_creator_agent,
    context=[curate_task],
    output_pydantic=Quiz,
)

project_task = Task(
    description=(
        "Recommend 3 practical projects for a {expertise_level} learner studying {topics}. "
        "First call the Project Suggestion Tool for each topic with expertise_level='{expertise_level}' "
        "to get the right scope and constraints, then write projects that fit that template. "
        "Projects should build on the curated learning materials."
    ),
    expected_output="A ProjectIdeas object with 3 projects, each with skills practiced and estimated hours.",
    agent=project_idea_agent,
    context=[curate_task],
    tools=[project_tool],
    output_pydantic=ProjectIdeas,
)
print("Tasks ready.")

Tasks ready.


---
## 8. Run the crew

`Process.sequential` runs the tasks in list order, each one seeing the outputs named in its
`context`. The verbose agent logs show the search calls, tool calls and final answers.

In [40]:
import os
import sys
import traceback
from IPython.utils import io
from crewai import LLM, Crew, Process

openai_key = os.environ["OPENAI_API_KEY"]
# Ensure API key is set
os.environ["OPENAI_API_KEY"] = openai_key
llm = LLM(model=MODEL, temperature=0.3, api_key=openai_key)

# Re-bind LLM to agents
learning_material_agent.llm = llm
quiz_creator_agent.llm = llm
project_idea_agent.llm = llm

# Turn off verbosity to prevent Colab output truncation
learning_material_agent.verbose = False
quiz_creator_agent.verbose = False
project_idea_agent.verbose = False

crew = Crew(
    agents=[learning_material_agent, quiz_creator_agent, project_idea_agent],
    tasks=[curate_task, quiz_task, project_task],
    process=Process.sequential,
    verbose=False,
    planning=False
)

try:
    # Silence all console output robustly
    with io.capture_output() as captured:
        result = await crew.kickoff_async(inputs={
            "topics": ", ".join(TOPICS),
            "expertise_level": EXPERTISE_LEVEL,
        })

    if hasattr(result, 'tasks_output'):
        print("\nDone. Tasks completed:", len(result.tasks_output))
    else:
        print("\nDone. Result:", result)
except Exception as e:
    print("\n--- EXECUTION FAILED ---")
    print(traceback.format_exc())



Done. Tasks completed: 3


---
## 9. Results

Each task's validated Pydantic object is on `result.tasks_output[i].pydantic`. The cells below
render them as readable output and then save everything to one JSON file.

In [41]:
materials: LearningMaterials = result.tasks_output[0].pydantic
quiz: Quiz = result.tasks_output[1].pydantic
projects: ProjectIdeas = result.tasks_output[2].pydantic

print("=" * 70)
print(f"LEARNING MATERIALS  ({materials.expertise_level}: {', '.join(materials.topics)})")
print("=" * 70)
print("\nVideos:")
for r in materials.videos:
    print(f"  - {r.title}\n    {r.url}\n    {r.why}")
print("\nArticles:")
for r in materials.articles:
    print(f"  - {r.title}\n    {r.url}\n    {r.why}")
print("\nExercises:")
for e in materials.exercises:
    print(f"  - {e.title}: {e.instructions}")

LEARNING MATERIALS  (Beginner: Python decorators, SQL joins)

Videos:
  - Python Decorators Explained
    https://www.youtube.com/watch?v=FsAPt_9Bf3I
    This video provides a clear and concise explanation of decorators in Python, making it suitable for beginners.
  - SQL Joins Tutorial
    https://www.youtube.com/watch?v=9yeOJ0ZMUaw
    This tutorial covers the basics of SQL joins with practical examples, ideal for beginners learning SQL.

Articles:
  - Understanding Decorators in Python
    https://realpython.com/primer-on-python-decorators/
    This article from Real Python offers a beginner-friendly introduction to decorators with examples and explanations.
  - SQL Joins Explained
    https://www.w3schools.com/sql/sql_join.asp
    W3Schools provides a straightforward explanation of SQL joins with visual aids, making it easy for beginners to understand.

Exercises:
  - Create a Simple Decorator: Write a decorator that prints 'Function is called' before executing any function. Test i

In [42]:
print("=" * 70)
print(f"QUIZ: {quiz.title}")
print("=" * 70)
for i, q in enumerate(quiz.questions, 1):
    print(f"\n{i}. {q.question}")
    for opt in q.options:
        print(f"     {opt}")
    print(f"   Answer: {q.correct_answer} - {q.explanation}")

QUIZ: Python Decorators and SQL Joins Quiz

1. What is the primary purpose of a decorator in Python?
     A) To modify the behavior of a function
     B) To create a new function
     C) To handle exceptions
     D) To define a class
   Answer: A - A decorator is used to modify or enhance the behavior of a function without changing its code.

2. Which SQL join returns all records from the left table and matched records from the right table?
     A) INNER JOIN
     B) LEFT JOIN
     C) RIGHT JOIN
     D) FULL JOIN
   Answer: B - A LEFT JOIN returns all records from the left table and the matched records from the right table, filling with NULLs where there are no matches.

3. How do you apply a decorator to a function in Python?
     A) By using the @ symbol above the function definition
     B) By calling the decorator function inside the function
     C) By importing the decorator module
     D) By defining the decorator inside the function
   Answer: A - In Python, decorators are appl

In [43]:
print("=" * 70)
print(f"PROJECT IDEAS  ({projects.expertise_level})")
print("=" * 70)
for i, p in enumerate(projects.projects, 1):
    print(f"\n{i}. {p.title}  (~{p.estimated_hours} h)")
    print(f"   {p.description}")
    print(f"   Skills: {', '.join(p.skills_practiced)}")

PROJECT IDEAS  (beginner)

1. Simple Decorator Implementation  (~3 h)
   Create a Python script that defines a simple decorator to log when a function is called. The decorator should print 'Function is called' before executing any function it decorates. Test it with a few sample functions to demonstrate its effectiveness.
   Skills: Understanding of Python decorators, Function definition and calling, Basic logging in Python

2. Basic SQL Joins Demonstration  (~3 h)
   Using a sample SQLite database, write a Python script that executes SQL queries to demonstrate INNER JOIN, LEFT JOIN, and RIGHT JOIN. The script should print the results of each query to show how the joins work.
   Skills: Writing SQL queries, Understanding SQL joins, Interacting with databases using Python

3. Combining Decorators with SQL Queries  (~4 h)
   Develop a Python script that uses a decorator to time how long SQL queries take to execute. The decorator should print the execution time before returning the result

In [44]:
import json

output = {
    "learner": {"topics": TOPICS, "expertise_level": EXPERTISE_LEVEL},
    "learning_materials": materials.model_dump(),
    "quiz": quiz.model_dump(),
    "project_ideas": projects.model_dump(),
}
with open("education_plan.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print("Saved education_plan.json")
print(json.dumps(output, indent=2)[:1500], "...")

Saved education_plan.json
{
  "learner": {
    "topics": [
      "Python decorators",
      "SQL joins"
    ],
    "expertise_level": "beginner"
  },
  "learning_materials": {
    "topics": [
      "Python decorators",
      "SQL joins"
    ],
    "expertise_level": "Beginner",
    "videos": [
      {
        "title": "Python Decorators Explained",
        "url": "https://www.youtube.com/watch?v=FsAPt_9Bf3I",
        "why": "This video provides a clear and concise explanation of decorators in Python, making it suitable for beginners."
      },
      {
        "title": "SQL Joins Tutorial",
        "url": "https://www.youtube.com/watch?v=9yeOJ0ZMUaw",
        "why": "This tutorial covers the basics of SQL joins with practical examples, ideal for beginners learning SQL."
      }
    ],
    "articles": [
      {
        "title": "Understanding Decorators in Python",
        "url": "https://realpython.com/primer-on-python-decorators/",
        "why": "This article from Real Python offers a

---
## Design notes

- **Sequential process** because every task depends on the one before it. A hierarchical process
  would add a manager agent to decide an order that is already fixed.
- **Pydantic outputs** turn free text into checked data: a quiz with three options or a project
  without an hours estimate fails validation instead of silently reaching the learner.
- **The custom tool holds the rule, the agent holds the creativity.** Mapping expertise level to
  project scope is a policy decision, so it lives in deterministic code; the model's job is to
  turn that scope into interesting projects.
- **Inputs are validated before kickoff** (allowed levels, topic count and length, no braces),
  and keys are never printed. The LLM's output and anything returned by web search are treated as
  untrusted content, which is why the structured outputs are validated rather than trusted.